In [1]:
# Part 3
from google.colab import files
uploaded = files.upload()

Saving Womens Clothing E-Commerce Reviews.csv to Womens Clothing E-Commerce Reviews (5).csv


In [12]:
# Task 1
# a:
zero_shot_template = """Classify the sentiment of this clothing review as positive, negative, or neutral.
Respond ONLY in this exact JSON format: {{"label": "positive|negative|neutral", "confidence": "low|medium|high", "reason": "string"}}

Review: {review}"""

print(zero_shot_template)

# b:
few_shot_template = """Classify the sentiment of this clothing review as positive, negative, or neutral.
Respond ONLY in this exact JSON format: {{"label": "positive|negative|neutral", "confidence": "low|medium|high", "reason": "string"}}

Here are some examples:

Review: "Love it, fits perfectly and the fabric feels amazing!"
{{"label": "positive", "confidence": "high", "reason": "praises fit and fabric quality"}}

Review: "Terrible quality, fell apart after one wash."
{{"label": "negative", "confidence": "high", "reason": "complains about poor quality"}}

Review: "It's okay, nothing special but not bad either."
{{"label": "neutral", "confidence": "medium", "reason": "mixed, unremarkable opinion"}}

Now classify this review:
Review: {review}"""

print(few_shot_template)

# c:
role_prompted_template = """Act as a senior customer-insights analyst who specializes in e-commerce fashion feedback.

Context: You are reviewing customer feedback for a women's clothing brand. Your job
is to accurately classify sentiment so the business can prioritize responses.

Constraint: Respond ONLY in this exact JSON format, with no extra text:
{{"label": "positive|negative|neutral", "confidence": "low|medium|high", "reason": "string"}}

Review: {review}"""

print(role_prompted_template)

Classify the sentiment of this clothing review as positive, negative, or neutral.
Respond ONLY in this exact JSON format: {{"label": "positive|negative|neutral", "confidence": "low|medium|high", "reason": "string"}}

Review: {review}
Classify the sentiment of this clothing review as positive, negative, or neutral.
Respond ONLY in this exact JSON format: {{"label": "positive|negative|neutral", "confidence": "low|medium|high", "reason": "string"}}

Here are some examples:

Review: "Love it, fits perfectly and the fabric feels amazing!"
{{"label": "positive", "confidence": "high", "reason": "praises fit and fabric quality"}}

Review: "Terrible quality, fell apart after one wash."
{{"label": "negative", "confidence": "high", "reason": "complains about poor quality"}}

Review: "It's okay, nothing special but not bad either."
{{"label": "neutral", "confidence": "medium", "reason": "mixed, unremarkable opinion"}}

Now classify this review:
Review: {review}
Act as a senior customer-insights an

In [13]:
# Task 2:
!pip install groq -q

from groq import Groq
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

def call_llm(prompt, temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

test_response = call_llm("Say hello in one sentence.")
print(test_response)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [14]:
# Task 3:
import time

def call_llm(prompt, temperature=0.7, max_tokens=500):
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2)
    print("All 3 attempts failed.")
    return None

test_response = call_llm("Say hello in one sentence.")
print(test_response)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [15]:
import pandas as pd
text_df = pd.read_csv('Womens Clothing E-Commerce Reviews.csv')
print(text_df.columns.tolist())
print(text_df.head())

['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']
   Unnamed: 0  Clothing ID  Age                    Title  \
0           0          767   33                      NaN   
1           1         1080   34                      NaN   
2           2         1077   60  Some major design flaws   
3           3         1049   50         My favorite buy!   
4           4          847   47         Flattering shirt   

                                         Review Text  Rating  Recommended IND  \
0  Absolutely wonderful - silky and sexy and comf...       4                1   
1  Love this dress!  it's sooo pretty.  i happene...       5                1   
2  I had such high hopes for this dress and reall...       3                0   
3  I love, love, love this jumpsuit. it's fun, fl...       5                1   
4  This shirt is very flattering to all due to th...       5       

In [16]:
# Task 4:
import json

sample_reviews = text_df['Review Text'].dropna().head(5).tolist()

templates = {
    'zero_shot': zero_shot_template,
    'few_shot': few_shot_template,
    'role_prompted': role_prompted_template
}

results = []

for i, review in enumerate(sample_reviews):
    for template_name, template_text in templates.items():
        prompt = template_text.format(review=review)
        raw_response = call_llm(prompt)

        try:
            parsed = json.loads(raw_response)
            valid = True
        except:
            parsed = raw_response
            valid = False

        results.append({
            'record': i+1,
            'template': template_name,
            'valid_json': valid,
            'parsed_output': parsed
        })

for r in results:
    print(r)

{'record': 1, 'template': 'zero_shot', 'valid_json': True, 'parsed_output': {'label': 'positive', 'confidence': 'high', 'reason': "The reviewer uses very positive adjectives such as 'wonderful', 'silky', 'sexy', and 'comfortable' to describe the clothing."}}
{'record': 1, 'template': 'few_shot', 'valid_json': True, 'parsed_output': {'label': 'positive', 'confidence': 'high', 'reason': 'uses strong positive adjectives to describe the product'}}
{'record': 1, 'template': 'role_prompted', 'valid_json': True, 'parsed_output': {'label': 'positive', 'confidence': 'high', 'reason': 'customer uses strong positive adjectives such as wonderful, silky, sexy, and comfortable to describe the product'}}
{'record': 2, 'template': 'zero_shot', 'valid_json': True, 'parsed_output': {'label': 'positive', 'confidence': 'high', 'reason': "The reviewer uses enthusiastic language such as 'Love this dress' and 'sooo pretty', and also mentions they would 'definitely' recommend it, indicating a strong positive 

In [17]:
# Task 5:
aspect_template = """Act as a senior customer-insights analyst who specializes in e-commerce fashion feedback.

Context: You are reviewing customer feedback for a women's clothing brand, focusing on
two specific aspects: product quality and fit.

Constraint: Respond ONLY in this exact JSON format, with no extra text:
{{
  "product_quality": {{"label": "positive|negative|neutral", "actionable_phrase": "3-6 word phrase"}},
  "fit": {{"label": "positive|negative|neutral", "actionable_phrase": "3-6 word phrase"}}
}}

Review: {review}"""

sample_10_reviews = text_df['Review Text'].dropna().head(10).tolist()

aspect_results = []
for i, review in enumerate(sample_10_reviews):
    prompt = aspect_template.format(review=review)
    raw_response = call_llm(prompt)
    try:
        parsed = json.loads(raw_response)
    except:
        parsed = raw_response
    aspect_results.append({'record': i+1, 'review_snippet': review[:60], 'result': parsed})

for r in aspect_results:
    print(r)

{'record': 1, 'review_snippet': 'Absolutely wonderful - silky and sexy and comfortable', 'result': {'product_quality': {'label': 'positive', 'actionable_phrase': 'Silky and sexy'}, 'fit': {'label': 'positive', 'actionable_phrase': 'Very comfortable fit'}}}
{'record': 2, 'review_snippet': "Love this dress!  it's sooo pretty.  i happened to find it i", 'result': {'product_quality': {'label': 'positive', 'actionable_phrase': 'Love this dress'}, 'fit': {'label': 'positive', 'actionable_phrase': 'Flattering length found'}}}
{'record': 3, 'review_snippet': 'I had such high hopes for this dress and really wanted it to', 'result': {'product_quality': {'label': 'negative', 'actionable_phrase': 'cheap materials used'}, 'fit': {'label': 'negative', 'actionable_phrase': 'runs very small'}}}
{'record': 4, 'review_snippet': "I love, love, love this jumpsuit. it's fun, flirty, and fabu", 'result': {'product_quality': {'label': 'positive', 'actionable_phrase': 'Great material used'}, 'fit': {'label': 

In [18]:
# Task 6:
response_template = """Act as a customer service representative for a women's clothing brand.

A customer left this feedback: "{review}"

Our analysis found: Product quality was {quality_label} ({quality_phrase}).
Fit was {fit_label} ({fit_phrase}).

Write a short, professional, empathetic reply (2-4 sentences) that directly addresses
these specific points. Do not use generic phrases like "thank you for your feedback"
without also referencing the specific issue or praise mentioned above."""

drafted_replies = []

for i in range(3):  # draft replies for first 3 records as examples
    review = sample_10_reviews[i]
    result = aspect_results[i]['result']

    if isinstance(result, dict):
        quality_label = result['product_quality']['label']
        quality_phrase = result['product_quality']['actionable_phrase']
        fit_label = result['fit']['label']
        fit_phrase = result['fit']['actionable_phrase']

        prompt = response_template.format(
            review=review,
            quality_label=quality_label,
            quality_phrase=quality_phrase,
            fit_label=fit_label,
            fit_phrase=fit_phrase
        )

        drafted_reply = call_llm(prompt)
        drafted_replies.append({'record': i+1, 'review_snippet': review[:60], 'drafted_reply': drafted_reply})

for r in drafted_replies:
    print(r)
    print('---')

{'record': 1, 'review_snippet': 'Absolutely wonderful - silky and sexy and comfortable', 'drafted_reply': "We're thrilled to hear that you're enjoying the silky texture and sexy style of our garment, as we take great care in selecting high-quality materials that exude confidence and sophistication. It's also wonderful to know that the fit is comfortable, as we strive to create pieces that not only look great but also feel amazing to wear. Your positive experience with the comfort and quality of our product is exactly what we aim for, and we're glad we could deliver that for you."}
---
{'record': 2, 'review_snippet': "Love this dress!  it's sooo pretty.  i happened to find it i", 'drafted_reply': "We're thrilled to hear that you fell in love with our dress and found a flattering fit, despite initially being hesitant due to the petite sizing. It's great that the length worked well for you, hitting just below the knee, and we appreciate your insight that it would be a true midi on someone

In [20]:
# Task 7:
conversation_history = [
    {"role": "user", "content": "I'm looking for feedback on a jumpsuit review. The customer said: 'I love, love, love this jumpsuit. Fun, flirty, and fabulous!' What's the sentiment?"}
]

# First call
response1 = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=conversation_history,
    temperature=0.7,
    max_tokens=200
)
reply1 = response1.choices[0].message.content
print("Turn 1 response:", reply1)

# Add the assistant's reply to the conversation history
conversation_history.append({"role": "assistant", "content": reply1})

# Second turn - referencing turn 1 without repeating the info
conversation_history.append({"role": "user", "content": "Based on that sentiment, write one short marketing tagline for this jumpsuit."})

response2 = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=conversation_history,
    temperature=0.7,
    max_tokens=100
)
reply2 = response2.choices[0].message.content
print("\nTurn 2 response:", reply2)

print("\nFull conversation history object:")
for msg in conversation_history:
    print(msg)

Turn 1 response: The sentiment of this review is extremely positive. The customer uses the word "love" three times, which emphasizes their strong affection for the jumpsuit. They also use additional positive adjectives like "fun", "flirty", and "fabulous" to describe the product, indicating that they are very satisfied with their purchase. Overall, the tone is enthusiastic and energetic, suggesting that the customer is highly impressed with the jumpsuit.

Turn 2 response: "Fall in love with the fun, flirty, and fabulous you!"

Full conversation history object:
{'role': 'user', 'content': "I'm looking for feedback on a jumpsuit review. The customer said: 'I love, love, love this jumpsuit. Fun, flirty, and fabulous!' What's the sentiment?"}
{'role': 'assistant', 'content': 'The sentiment of this review is extremely positive. The customer uses the word "love" three times, which emphasizes their strong affection for the jumpsuit. They also use additional positive adjectives like "fun", "fl